# 🕸️ Notebook 3: Medical Books Knowledge Graph (KG)
**Goal:** Build a Knowledge Graph (Symptom → Disease → Treatment) by reading from actual Medical Textbooks (PDFs).

**Why:** Using real textbooks prevents hallucinations and grounds our QA system in verified medical literature.

**Input:** PDF files inside `data/books/`
**Output:** `data/kg_medical_books.pkl`

In [ ]:
!pip install -q PyMuPDF networkx matplotlib spacy
import fitz  # PyMuPDF
import os, glob, re
import networkx as nx
import matplotlib.pyplot as plt
import pickle
from collections import defaultdict

os.makedirs('data/books', exist_ok=True)
print('✅ Libraries loaded and data/books directory ready.')

## 1. Extract Text from Medical Textbooks (PDFs)

In [ ]:
pdf_files = glob.glob('data/books/*.pdf')
book_texts = []

if not pdf_files:
    print("⚠️ No PDF books found in 'data/books/'.")
    print("Using a sample fallback text for demonstration so the graph builds successfully.")
    print("Please upload your medical PDFs (English or Gujarati) to 'data/books/' and re-run this notebook.")
    sample_text = """Diabetes mellitus causes symptoms like frequent urination, excessive thirst, and fatigue. 
    Treatment for diabetes includes Metformin, insulin therapy, and dietary changes.
    Hypertension (high blood pressure) symptoms include headaches and shortness of breath, treated with Amlodipine.
    Malaria causes severe fever and chills; it is treated with Chloroquine or Artemisinin.
    ડેન્ગ્યુ (Dengue) ના લક્ષણો તાવ (fever) અને સાંધાનો દુખાવો છે, Paracetamol આપી શકાય.
    """
    book_texts.append(sample_text)
else:
    print(f"Found {len(pdf_files)} PDF(s). Extracting text...")
    for file_path in pdf_files:
        print(f"Reading: {os.path.basename(file_path)}")
        doc = fitz.open(file_path)
        text = ""
        for page in doc:
            text += page.get_text() + "\n"
        book_texts.append(text)
        
full_corpus = "\n".join(book_texts)
print(f"✅ Extracted {len(full_corpus)} characters of medical text.")

## 2. Entity Dictionaries (Bilingual Context)

In [ ]:
# Expanded dictionary covering both English and Gujarati (since PDFs could be in either)
SYMPTOMS = {
    'fever': 'તાવ (Fever)', 'તાવ': 'તાવ (Fever)',
    'headache': 'માથાનો દુખાવો', 'માથાનો દુખાવો': 'માથાનો દુખાવો',
    'cough': 'ખાંસી', 'ખાંસી': 'ખાંસી',
    'fatigue': 'થાક', 'થાક': 'થાક',
    'pain': 'દુખાવો', 'દુખાવો': 'દુખાવો',
    'chest pain': 'છાતીમાં દુખાવો', 'છાતીમાં દુખાવો': 'છાતીમાં દુખાવો',
    'urination': 'વારંવાર પેશાબ', 'thirst': 'વધુ તરસ', 'chills': 'ઠંડી લાગવી',
    'shortness of breath': 'શ્વાસ લેવામાં તકલીફ'
}

DISEASES = {
    'diabetes': 'ડાયાબિટીઝ (Diabetes)', 'ડાયાબિટીઝ': 'ડાયાબિટીઝ (Diabetes)',
    'malaria': 'મેલેરિયા (Malaria)', 'મેલેરિયા': 'મેલેરિયા (Malaria)',
    'dengue': 'ડેન્ગ્યુ (Dengue)', 'ડેન્ગ્યુ': 'ડેન્ગ્યુ (Dengue)',
    'hypertension': 'હાઈ બ્લડ પ્રેશર (Hypertension)', 'high blood pressure': 'હાઈ બ્લડ પ્રેશર (Hypertension)',
    'blood pressure': 'હાઈ બ્લડ પ્રેશર (Hypertension)',
    'stroke': 'સ્ટ્રોક (Stroke)', 'હાર્ટ એટેક': 'હાર્ટ એટેક (Heart Attack)',
    'asthma': 'અસ્થમા (Asthma)'
}

TREATMENTS = {
    'paracetamol': 'પેરાસિટામોલ (Paracetamol)', 'પેરાસિટામોલ': 'પેરાસિટામોલ (Paracetamol)',
    'metformin': 'મેટફોર્મિન (Metformin)', 'insulin': 'ઇન્સ્યુલિન (Insulin)',
    'chloroquine': 'ક્લોરોક્વિન (Chloroquine)', 'artemisinin': 'આર્ટેમિસિનિન (Artemisinin)',
    'amlodipine': 'એમ્લોડિપિન (Amlodipine)',
    'rest': 'આરામ (Rest)', 'આરામ': 'આરામ (Rest)', 
    'diet': 'આહારમાં ફેરફાર (Diet)'
}

print('✅ Entity dictionaries compiled.')

## 3. Extract Relationships from Books (Co-occurrence)

In [ ]:
edges_symptom_disease = defaultdict(int)
edges_disease_treatment = defaultdict(int)

# Split text into roughly sentence/paragraph chunks for context windowing
chunks = [c.strip().lower() for c in re.split(r'[\.\n]', full_corpus) if len(c.strip()) > 10]

def find_entities(text: str, entity_dict: dict):
    found = set()
    for kw, standard_name in entity_dict.items():
        if kw in text:
            found.add(standard_name)
    return found

for chunk in chunks:
    syms = find_entities(chunk, SYMPTOMS)
    diss = find_entities(chunk, DISEASES)
    trts = find_entities(chunk, TREATMENTS)
    
    # Symptom -> Disease links
    for s in syms:
        for d in diss:
            edges_symptom_disease[(s, d)] += 1
            
    # Disease -> Treatment links
    for d in diss:
        for t in trts:
            edges_disease_treatment[(d, t)] += 1

print(f"Extracted {len(edges_symptom_disease)} Symptom→Disease links.")
print(f"Extracted {len(edges_disease_treatment)} Disease→Treatment links.")

## 4. Build and Visualize Knowledge Graph

In [ ]:
G = nx.DiGraph()

# Add nodes with types
for (s, d), weight in edges_symptom_disease.items():
    if weight > 0:
        G.add_node(s, type='symptom')
        G.add_node(d, type='disease')
        G.add_edge(s, d, weight=weight, relation='indicates')

for (d, t), weight in edges_disease_treatment.items():
    if weight > 0:
        G.add_node(d, type='disease')
        G.add_node(t, type='treatment')
        G.add_edge(d, t, weight=weight, relation='treated_with')

print(f"✅ Knowledge Graph built from text books with {len(G.nodes)} nodes and {len(G.edges)} edges.")

# Visualize
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=0.8, seed=42)

# Colors by node type
colors = []
for node, data in G.nodes(data=True):
    ntype = data.get('type')
    if ntype == 'symptom': colors.append('#fca5a5')    # pastel red
    elif ntype == 'disease': colors.append('#fcd34d')  # pastel yellow
    elif ntype == 'treatment': colors.append('#86efac')# pastel green
    else: colors.append('lightgrey')

nx.draw(
    G, pos, with_labels=True,
    node_color=colors, node_size=2500,
    font_size=8, font_family='sans-serif',
    edge_color='gray', arrows=True, width=1.5
)

plt.title('Medical Book Knowledge Graph (Symptom → Disease → Treatment)', fontsize=14)
plt.show()

## 5. Export Graph for RAG

In [ ]:
with open('data/kg_medical_books.pkl', 'wb') as f:
    pickle.dump(G, f)

print('✅ Medical Book Knowledge Graph saved to `data/kg_medical_books.pkl`')
print('\n📝 NEXT STEP: Add PDFs to data/books/, then run Notebook 06 to build the Vector DB.')